# 📘 Notebook 2: Constructors & the `__init__` Method

---

## 🎯 Learning Objectives

By the end of this notebook, you will be able to:

1. Understand the **constructor** concept and Python's `__init__` method
2. Use `__init__` to **initialize** objects with required attributes
3. Add **type hints** to constructor parameters
4. Set **default parameter values**
5. Add **validations** using `assert` statements
6. Understand the difference between `__init__` and `__new__`

---

## 1. The Problem: Why Do We Need `__init__`?

In the previous notebook, we assigned attributes dynamically:

```python
item1 = Item()
item1.name = "Phone"    # What if we forget this?
item1.price = 100       # Or this?
item1.quantity = 5      # Or this?
```

### Problems with this approach:

| Problem | Consequence |
|---------|-------------|
| No enforcement | You can forget to set required attributes |
| No validation | You can set `price = -50` (a negative price!) |
| No consistency | Different objects might have different attributes |
| Verbose | You need multiple lines per object |

The **`__init__`** method solves all of these!

---

## 2. What is `__init__`?

**`__init__`** is a special method (also called a **dunder method** or **magic method**) that Python calls **automatically** when you create a new instance.

The name stands for **"initialize"** — it sets up the object's initial state.

```
item1 = Item("Phone", 100, 5)
         │      │      │   │
         │      └──────┴───┴── These arguments are passed to __init__
         │
         └── Python calls Item.__init__(new_object, "Phone", 100, 5)
```

### What happens when you write `Item("Phone", 100, 5)`:

1. Python creates a new empty `Item` object in memory
2. Python calls `__init__(self, "Phone", 100, 5)` where `self` is the new object
3. `__init__` sets up the object's attributes
4. The initialized object is returned and assigned to your variable

In [ ]:
# Our first __init__ method
class Item:
    def __init__(self, name, price, quantity):
        self.name = name
        self.price = price
        self.quantity = quantity
    
    def calculate_total_price(self):
        return self.price * self.quantity

# Now we MUST provide name, price, and quantity
item1 = Item("Phone", 100, 5)
item2 = Item("Laptop", 1000, 3)

print(f"{item1.name}: ${item1.calculate_total_price()}")
print(f"{item2.name}: ${item2.calculate_total_price()}")

In [ ]:
# What happens if we forget a required argument?
try:
    item3 = Item("Tablet", 500)  # Missing 'quantity'!
except TypeError as e:
    print(f"❌ Error: {e}")
    print("__init__ enforces that all required arguments are provided!")

---

## 3. Understanding `self.attribute = parameter`

This is the most important line pattern in `__init__`:

```python
def __init__(self, name, price, quantity):
    self.name = name
    #    │        │
    #    │        └── The parameter passed in (local variable)
    #    └── The attribute saved on the object (persists forever)
```

The **left side** (`self.name`) creates an **attribute on the object**.
The **right side** (`name`) is the **parameter value** passed to `__init__`.

They don't have to have the same name, but it's conventional:

In [ ]:
# The names DON'T have to match (but they should by convention)
class Item:
    def __init__(self, n, p, q):
        self.name = n       # attribute 'name' gets value of parameter 'n'
        self.price = p      # attribute 'price' gets value of parameter 'p'
        self.quantity = q   # attribute 'quantity' gets value of parameter 'q'

item = Item("Phone", 100, 5)
print(f"Name: {item.name}, Price: {item.price}, Qty: {item.quantity}")

# This works, but using matching names (self.name = name) is much clearer!

---

## 4. Adding Type Hints

**Type hints** are annotations that tell developers (and IDEs) what type each parameter should be. They **don't enforce** types at runtime, but they improve readability and IDE support.

```python
def __init__(self, name: str, price: float, quantity: int):
                        ^^^          ^^^^^            ^^^
                    Expected types (documentation only)
```

In [ ]:
class Item:
    def __init__(self, name: str, price: float, quantity: int):
        self.name = name
        self.price = price
        self.quantity = quantity
    
    def calculate_total_price(self) -> float:
        """Calculate and return the total price."""
        return self.price * self.quantity

item1 = Item("Phone", 100.0, 5)
print(f"Total: ${item1.calculate_total_price()}")

# Type hints don't prevent wrong types — they're documentation
item2 = Item(123, "not a number", [])  # This runs, but it's wrong!
print(f"Name is: {item2.name} (type: {type(item2.name).__name__})")
print("⚠️ Type hints don't enforce types — they're for documentation & IDE support")

---

## 5. Default Parameter Values

You can give parameters **default values** to make them optional. Parameters with defaults must come **after** required parameters.

```python
def __init__(self, name: str, price: float, quantity: int = 0):
                                                          ^^^
                                              Default value — quantity is optional
```

In [ ]:
class Item:
    def __init__(self, name: str, price: float, quantity: int = 0):
        self.name = name
        self.price = price
        self.quantity = quantity
    
    def calculate_total_price(self) -> float:
        return self.price * self.quantity

# With all arguments
item1 = Item("Phone", 100, 5)
print(f"{item1.name}: quantity={item1.quantity}, total=${item1.calculate_total_price()}")

# Without quantity — uses default of 0
item2 = Item("Laptop", 1000)
print(f"{item2.name}: quantity={item2.quantity}, total=${item2.calculate_total_price()}")

# Using keyword arguments (named arguments)
item3 = Item(name="Tablet", price=500, quantity=2)
print(f"{item3.name}: quantity={item3.quantity}, total=${item3.calculate_total_price()}")

---

## 6. Adding Validation with `assert`

What if someone passes a **negative price** or **negative quantity**? We should validate inputs!

The `assert` statement checks a condition and raises an `AssertionError` if it's `False`:

```python
assert condition, "Error message if condition is False"
```

In [ ]:
class Item:
    def __init__(self, name: str, price: float, quantity: int = 0):
        # Run validations BEFORE assigning attributes
        assert price >= 0, f"Price {price} is not greater than or equal to zero!"
        assert quantity >= 0, f"Quantity {quantity} is not greater than or equal to zero!"
        
        # Assign to self object (only reached if assertions pass)
        self.name = name
        self.price = price
        self.quantity = quantity
    
    def calculate_total_price(self) -> float:
        return self.price * self.quantity

# Valid items
item1 = Item("Phone", 100, 5)
print(f"✅ {item1.name} created successfully")

# Invalid item — negative price
try:
    item2 = Item("Scam", -50, 1)
except AssertionError as e:
    print(f"❌ AssertionError: {e}")

# Invalid item — negative quantity
try:
    item3 = Item("Ghost", 100, -3)
except AssertionError as e:
    print(f"❌ AssertionError: {e}")

> 💡 **Best Practice**: Put validations at the **top** of `__init__`, before any attribute assignments. This ensures invalid objects are never created.

> ⚠️ **Note**: `assert` statements can be disabled with the `-O` (optimize) flag when running Python. For production code, consider using `raise ValueError(...)` instead. We'll discuss this more in later notebooks.

---

## 7. Adding Computed Attributes in `__init__`

You can also create attributes that are **computed** from the input parameters:

In [ ]:
class Item:
    def __init__(self, name: str, price: float, quantity: int = 0):
        assert price >= 0, f"Price {price} is not >= 0!"
        assert quantity >= 0, f"Quantity {quantity} is not >= 0!"
        
        self.name = name
        self.price = price
        self.quantity = quantity
        
        # Computed attribute — derived from other values
        self.total_price = price * quantity

item = Item("Phone", 100, 5)
print(f"{item.name}: ${item.total_price}")

# ⚠️ Warning: computed attributes can become stale!
item.price = 200  # Changed the price...
print(f"Price changed to ${item.price}, but total_price is still ${item.total_price}")
print("This is why methods are better for computed values — they always recalculate!")

---

## 8. `__init__` vs `__new__` (Advanced)

For completeness, there are actually **two** steps in object creation:

| Method | Purpose | When called |
|--------|---------|-------------|
| `__new__` | **Creates** the object in memory | First — allocates memory |
| `__init__` | **Initializes** the object's attributes | Second — sets up state |

You almost **never** need to override `__new__`. It's used in advanced patterns like:
- Singleton pattern (ensuring only one instance exists)
- Immutable types (like `int`, `str`, `tuple`)
- Metaclasses

In [ ]:
# Demonstrating __new__ vs __init__ (for understanding only)
class Item:
    def __new__(cls, *args, **kwargs):
        print(f"1. __new__ called — creating object of type {cls.__name__}")
        instance = super().__new__(cls)
        return instance
    
    def __init__(self, name: str, price: float):
        print(f"2. __init__ called — initializing with name={name}, price={price}")
        self.name = name
        self.price = price

print("Creating an item:")
item = Item("Phone", 100)
print(f"3. Done! item.name = {item.name}")

---

## 9. The `__repr__` Method (Bonus)

When you print an object, you get an ugly memory address by default. The `__repr__` method lets you control what gets printed:

In [ ]:
# Without __repr__
class ItemBad:
    def __init__(self, name, price):
        self.name = name
        self.price = price

print(f"Without __repr__: {ItemBad('Phone', 100)}")

# With __repr__
class ItemGood:
    def __init__(self, name, price):
        self.name = name
        self.price = price
    
    def __repr__(self):
        return f"Item('{self.name}', {self.price})"

print(f"With __repr__:    {ItemGood('Phone', 100)}")

---

## 🏋️ Practice Exercises

### Exercise 1: `BankAccount` class
Create a `BankAccount` class with:
- `__init__` that takes `owner` (str) and `balance` (float, default 0)
- Validation: balance must be >= 0
- A `deposit(amount)` method that adds to the balance
- A `withdraw(amount)` method that subtracts from the balance (but can't go below 0)

In [ ]:
# Exercise 1: Your code here



### Exercise 2: `Circle` class
Create a `Circle` class with:
- `__init__` that takes `radius` (float) with validation (must be > 0)
- A `area()` method (π × r²)
- A `circumference()` method (2 × π × r)
- A `__repr__` method

In [ ]:
# Exercise 2: Your code here



### Exercise 3: `Temperature` class
Create a `Temperature` class with:
- `__init__` that takes `celsius` (float)
- A `to_fahrenheit()` method
- A `to_kelvin()` method
- Validation: celsius must be >= -273.15 (absolute zero)

In [ ]:
# Exercise 3: Your code here



---

### ✅ Solutions

In [ ]:
# Solution 1: BankAccount
class BankAccount:
    def __init__(self, owner: str, balance: float = 0):
        assert balance >= 0, f"Initial balance {balance} cannot be negative!"
        self.owner = owner
        self.balance = balance
    
    def deposit(self, amount: float):
        assert amount > 0, "Deposit amount must be positive!"
        self.balance += amount
        print(f"Deposited ${amount}. New balance: ${self.balance}")
    
    def withdraw(self, amount: float):
        assert amount > 0, "Withdrawal amount must be positive!"
        if amount > self.balance:
            print(f"❌ Insufficient funds! Balance: ${self.balance}, Requested: ${amount}")
        else:
            self.balance -= amount
            print(f"Withdrew ${amount}. New balance: ${self.balance}")
    
    def __repr__(self):
        return f"BankAccount('{self.owner}', balance={self.balance})"

# Test it
account = BankAccount("Alice", 1000)
account.deposit(500)
account.withdraw(200)
account.withdraw(5000)  # Should fail
print(account)

In [ ]:
# Solution 2: Circle
import math

class Circle:
    def __init__(self, radius: float):
        assert radius > 0, f"Radius {radius} must be positive!"
        self.radius = radius
    
    def area(self) -> float:
        return math.pi * self.radius ** 2
    
    def circumference(self) -> float:
        return 2 * math.pi * self.radius
    
    def __repr__(self):
        return f"Circle(radius={self.radius})"

c = Circle(5)
print(f"{c} → Area: {c.area():.2f}, Circumference: {c.circumference():.2f}")

In [ ]:
# Solution 3: Temperature
class Temperature:
    def __init__(self, celsius: float):
        assert celsius >= -273.15, f"Temperature {celsius}°C is below absolute zero!"
        self.celsius = celsius
    
    def to_fahrenheit(self) -> float:
        return (self.celsius * 9/5) + 32
    
    def to_kelvin(self) -> float:
        return self.celsius + 273.15
    
    def __repr__(self):
        return f"Temperature({self.celsius}°C)"

temp = Temperature(100)
print(f"{temp} = {temp.to_fahrenheit():.1f}°F = {temp.to_kelvin():.1f}K")

# Test absolute zero validation
try:
    impossible = Temperature(-300)
except AssertionError as e:
    print(f"❌ {e}")

---

## 📌 Key Takeaways

1. **`__init__`** is called automatically when creating an object — it initializes attributes
2. **`self.attribute = parameter`** saves data on the object for later use
3. **Type hints** improve readability but don't enforce types at runtime
4. **Default values** make parameters optional: `quantity: int = 0`
5. **`assert`** statements validate inputs and prevent invalid objects from being created
6. **`__repr__`** controls how your object is displayed when printed

---

**⏭️ Next: [Notebook 03 — Class Attributes & Instance Attributes](./03_Class_and_Instance_Attributes.ipynb)** — Learn the difference between data shared by all instances vs. data unique to each instance!